# NB05 — Airbnb Dashboard Data Export

**Technologies:** Microsoft Fabric | PySpark | Pandas | CSV | Parquet  
**Data Layer:** Gold → Dashboard Export | Silver & Gold → Backup

## 1. Objective

Prepare validated Airbnb data for **dashboard reporting** and export Silver and Gold tables for backup and future analysis.

The notebook creates two reporting datasets and preserves the processed tables in CSV and Parquet formats.

## 2. Data Sources & Outputs

| Type | Source |
|---|---|
| Gold Inputs | `dbo.dim_listing`, `dbo.dim_date`, `dbo.fact_availability` |
| Dashboard Export | `airbnb_listing_summary.csv` |
| Dashboard Export | `airbnb_daily_availability.csv` |
| Data Backup | 3 Silver + 4 Gold tables |

**Export Locations:**
- Dashboard: `Files/dashboard_exports/`
- Backup: `Files/airbnb_full_backup/`

## 3. Data Grain

| Dataset | Data Grain |
|---|---|
| Listing Summary | One row per listing |
| Daily Availability | One row per date, LGA and room type |

**Design Decision:** Keep listing-level details for supply and pricing analysis while aggregating availability data to reduce dashboard processing requirements.

## 4. Export Process

1. **Read Gold Tables:** Load Listing, Date and Availability datasets.
2. **Listing Export:** Select listing attributes, LGA, room type, indicative prices and quote-validity fields.
3. **Availability Aggregation:** Join Fact and Dimension tables, then group by `date`, `lga` and `room_type`.
4. **Data Validation:** Check unique Listing IDs, join row counts and aggregated totals.
5. **Dashboard Export:** Save two UTF-8 CSV files to OneLake.
6. **Data Backup:** Export all seven Silver and Gold tables as CSV and Parquet.

## 5. Aggregation & Business Logic

The Daily Availability dataset contains:

| Field | Meaning |
|---|---|
| `date` | Calendar date |
| `lga` | Local Government Area |
| `room_type` | Accommodation category |
| `available_listing_days` | Count of records advertised as available |
| `total_listing_days` | Total Listing-Date records |

**Advertised Availability Rate:**

`SUM(available_listing_days) / SUM(total_listing_days)`

The calculation uses aggregated counts rather than averaging individual group percentages.

**Limitation:** Advertised availability does not represent confirmed bookings or actual occupancy.

## 6. Validated Output

| Metric | Result |
|---|---:|
| Listing Summary Rows | 25,715 |
| Daily Availability Groups | 7,470 |
| Original Fact Records | 2,314,350 |
| Aggregated Listing-Days | 2,314,350 |
| Silver Tables Backed Up | 3 |
| Gold Tables Backed Up | 4 |
| Backup Formats | CSV + Parquet |

**Result:** Both dashboard CSVs and all seven table backups were exported successfully.

## 7. Pipeline Dependency & Export Notes

**NB01–NB03 Silver ETL → NB04 Gold → NB05 Export**

NB05 requires completed Gold tables before preparing dashboard datasets.

Dashboard CSVs are exported through Pandas after row-count safety checks. Full-table backups use PySpark and are saved separately in CSV and Parquet formats.

The notebook prepares files for external reporting; it does not automatically publish or refresh a Looker Studio dashboard.

In [ ]:
# Import Required Libraries
from pyspark.sql import functions as F
import os

StatementMeta(, 2caf78ea-bcb8-4968-99bf-b09d67e7bcf2, 3, Finished, Available, Finished, False)

In [1]:
# Read Gold tables
dim_listing = spark.table("dbo.dim_listing")
dim_date = spark.table("dbo.dim_date")
fact = spark.table("dbo.fact_availability")

# One row per Airbnb listing
listing_export = (
    dim_listing
    .select(
        "listing_id",
        "lga",
        "room_type",
        "listing_price_aud",
        "quote_price_aud",
        "has_valid_quote",
        "quote_checkin_date",
        "quote_checkout_date"
    )
    .withColumn(
        "room_type",
        F.coalesce(F.col("room_type"), F.lit("Unknown"))
    )
)

# Validate listing grain
total_listings = listing_export.count()
unique_listings = (
    listing_export.select("listing_id").distinct().count()
)

assert total_listings > 0
assert total_listings == unique_listings, \
    "Duplicate listing IDs!"

print("Listing Export Rows:", total_listings)

display(listing_export.limit(5))


StatementMeta(, c83cd5b1-4e84-4462-970f-4c3aff41877a, 3, Finished, Available, Finished, False)

Listing Export Rows: 25715


SynapseWidget(Synapse.DataFrame, 26f1d088-0061-45ec-830d-56bd9678f455)

In [2]:

# Join Fact with Listing and Date dimensions

availability_detail = (
    fact
    .join(
        listing_export.select(
            "listing_id", "lga", "room_type"
        ),
        on="listing_id",
        how="inner"
    )
    .join(
        dim_date.select("date_key", "date"),
        on="date_key",
        how="inner"
    )
)

# Validate no records were lost in the joins
fact_rows = fact.count()
joined_rows = availability_detail.count()

assert joined_rows == fact_rows, \
    "Fact rows were lost or duplicated!"

# Aggregate by date, LGA and room type
daily_availability = (
    availability_detail
    .groupBy("date", "lga", "room_type")
    .agg(
        F.sum("is_available").alias("available_listing_days"),
        F.count("*").alias("total_listing_days")
    )
    .orderBy("date", "lga", "room_type")
)

# Check aggregated counts
aggregated_rows = daily_availability.agg(
    F.sum("total_listing_days")
).first()[0]

assert aggregated_rows == fact_rows, \
    "Aggregated row counts do not match!"

print("Original Fact Rows:", fact_rows)
print("Aggregated Groups:", daily_availability.count())
print("Total Listing-Days:", aggregated_rows)

display(daily_availability.limit(10))


StatementMeta(, c83cd5b1-4e84-4462-970f-4c3aff41877a, 4, Finished, Available, Finished, False)

Original Fact Rows: 2314350
Aggregated Groups: 7470
Total Listing-Days: 2314350


SynapseWidget(Synapse.DataFrame, 41b9ae4b-2b14-49d9-87d2-65fe10540eec)

In [3]:
# Output folder in the default Lakehouse
export_dir = "/lakehouse/default/Files/dashboard_exports"

os.makedirs(export_dir, exist_ok=True)

# Safety checks before converting to Pandas
assert total_listings < 100000
assert daily_availability.count() < 100000

# Export Listing Summary
listing_path = f"{export_dir}/airbnb_listing_summary.csv"

listing_export.toPandas().to_csv(
    listing_path,
    index=False,
    encoding="utf-8-sig"
)

# Export Daily Availability Summary
availability_path = f"{export_dir}/airbnb_daily_availability.csv"

daily_availability.toPandas().to_csv(
    availability_path,
    index=False,
    encoding="utf-8-sig"
)

print("Listing CSV exported successfully!")
print("Daily Availability CSV exported successfully!")

print("Files:")
print(listing_path)
print(availability_path)


StatementMeta(, c83cd5b1-4e84-4462-970f-4c3aff41877a, 5, Finished, Available, Finished, False)

Listing CSV exported successfully!
Daily Availability CSV exported successfully!
Files:
/lakehouse/default/Files/dashboard_exports/airbnb_listing_summary.csv
/lakehouse/default/Files/dashboard_exports/airbnb_daily_availability.csv


In [2]:

# Export ALL Silver and Gold tables to CSV and Parquet
# Microsoft Fabric - LakeHouse_Airbnb

# Tables to export
tables = {
    "Silver": [
        "silver_listings",
        "silver_neighbourhoods",
        "silver_calendar_90d"
    ],
    "Gold": [
        "dim_neighbourhood",
        "dim_listing",
        "dim_date",
        "fact_availability"
    ]
}

# Destination in OneLake
base_path = "Files/airbnb_full_backup"

def export_one_file(df, layer, table_name, file_format):
    """
    Export one table as a single CSV or Parquet file.
    """
    extension = "csv" if file_format == "csv" else "parquet"

    output_folder = f"{base_path}/{layer}"
    temp_folder = f"{output_folder}/_temp_{table_name}_{extension}"
    final_path = f"{output_folder}/{table_name}.{extension}"

    notebookutils.fs.mkdirs(output_folder)

    writer = (
        df.coalesce(1)
        .write
        .format(file_format)
        .mode("overwrite")
    )

    if file_format == "csv":
        writer = writer.option("header", "true")

    # Spark first creates a directory with a part file
    writer.save(temp_folder)

    # Find the actual data file
    part_files = [
        f for f in notebookutils.fs.ls(temp_folder)
        if f.name.startswith("part-")
        and f.name.endswith(f".{extension}")
    ]

    if len(part_files) != 1:
        raise RuntimeError(
            f"Expected one {extension} file for {table_name}"
        )

    # Move and rename the file
    moved = notebookutils.fs.mv(
        part_files[0].path,
        final_path,
        create_path=True,
        overwrite=True
    )

    if not moved or not notebookutils.fs.exists(final_path):
        raise RuntimeError(
            f"Export failed: {final_path}"
        )

    # Remove temporary Spark directory
    notebookutils.fs.rm(temp_folder, True)

    print(f"  Saved: {final_path}")


# Export all 7 tables
for layer, table_names in tables.items():

    print(f"\n===== {layer.upper()} LAYER =====")

    for table_name in table_names:

        print(f"\nExporting: {table_name}")

        # Read existing Delta table
        df = spark.table(f"dbo.{table_name}")

        # Verify data exists
        row_count = df.count()
        if row_count == 0:
            raise ValueError(f"Empty table: {table_name}")

        print(f"Rows: {row_count:,}")

        # Export both formats
        export_one_file(df, layer, table_name, "csv")
        export_one_file(df, layer, table_name, "parquet")

        print(f"SUCCESS: {table_name}")

print("\nALL SILVER AND GOLD TABLES EXPORTED!")


StatementMeta(, 2caf78ea-bcb8-4968-99bf-b09d67e7bcf2, 4, Finished, Available, Finished, False)


===== SILVER LAYER =====

Exporting: silver_listings
Rows: 25,715
  Saved: Files/airbnb_full_backup/Silver/silver_listings.csv
  Saved: Files/airbnb_full_backup/Silver/silver_listings.parquet
SUCCESS: silver_listings

Exporting: silver_neighbourhoods
Rows: 30
  Saved: Files/airbnb_full_backup/Silver/silver_neighbourhoods.csv
  Saved: Files/airbnb_full_backup/Silver/silver_neighbourhoods.parquet
SUCCESS: silver_neighbourhoods

Exporting: silver_calendar_90d
Rows: 2,314,350
  Saved: Files/airbnb_full_backup/Silver/silver_calendar_90d.csv
  Saved: Files/airbnb_full_backup/Silver/silver_calendar_90d.parquet
SUCCESS: silver_calendar_90d

===== GOLD LAYER =====

Exporting: dim_neighbourhood
Rows: 30
  Saved: Files/airbnb_full_backup/Gold/dim_neighbourhood.csv
  Saved: Files/airbnb_full_backup/Gold/dim_neighbourhood.parquet
SUCCESS: dim_neighbourhood

Exporting: dim_listing
Rows: 25,715
  Saved: Files/airbnb_full_backup/Gold/dim_listing.csv
  Saved: Files/airbnb_full_backup/Gold/dim_listing.